In [45]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset,DataLoader
import matplotlib.pyplot as plt

In [46]:
## ADDED for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [47]:
torch.manual_seed(42)

In [48]:
df = pd.read_csv('/content/fashion-mnist_train.csv')
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [49]:
df.shape

(60000, 785)

In [ ]:
# Create a 4X4 grid of image
fig, axes = plt.subplots(4,4, figsize=(10,10))
fig.suptitle("First 16 images", fontsize=16)

# Plot the first 16 iamges form the dataset
for i, ax in enumerate(axes.flat):
  img = df.iloc[i,1:].values.reshape(28,28)
  ax.imshow(img)
  ax.axis('off')
  ax.set_title(f"Label: {df.iloc[i,0]}")
plt.tight_layout(rect=[0,0,1,0.96])
plt.show()
# print(fig)
# print(type(fig))
# print(axes)
# print((axes))

In [ ]:
# Train test split
X = df.iloc[:,1:,].values
y = df.iloc[:,0,].values
print(type(X))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Feature Scaling
X_train = X_train / 255.0
X_test = X_test / 255.0

In [ ]:
# CREATE CustomDataset class
class CustomDataset(Dataset):
  def __init__(self, features, labels):
    self.features = torch.tensor(features, dtype=torch.float32)
    self.labels = torch.tensor(labels, dtype=torch.long)

  def __len__(self):
    return len(self.features)

  def __getitem__(self, idx):
    return self.features[idx], self.labels[idx]

In [ ]:
# CREATE TRAIN_DATASET OBJECT
train_dataset = CustomDataset(X_train, y_train)

In [ ]:
train_dataset[0]
len(train_dataset)

In [ ]:
# CREATE TEST DATASET OBJECT
test_dataset = CustomDataset(X_test, y_test)
len(test_dataset)

In [ ]:
# def CustomDataLoader(dataset, batch_size, shuffle=True):
#   return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

In [ ]:
# Create the train and test loader

## ADDED for GPU
# batch_size=128, pin_memory = True
# The dataloader has to be moved inside the
# objecive function to hyperparameter tune it as well

In [ ]:
# train_loader

In [ ]:
class MyNN(nn.Module):
  def __init__(self, input_dim, out_dim, num_hidden_layers, neurons_per_layer, dropout_rate):
    super().__init__()
    layers = []
    for x in range(num_hidden_layers):
      layers.append(nn.Linear(input_dim, neurons_per_layer))
      layers.append(nn.BatchNorm1d(neurons_per_layer))
      layers.append(nn.ReLU())
      layers.append(nn.Dropout(dropout_rate))
      input_dim = neurons_per_layer
    layers.append(nn.Linear(input_dim, out_dim))
    self.model = nn.Sequential(*layers)
        # Unpacking the list since Sequential requires layer one by one

  def forward(self, x):
    return self.model(x)


In [ ]:
# Building an Objective function
def objective_function(trial):

  # Hyperparameter extraction for the next trial from the trial's searchspace
  num_hidden_layers = trial.suggest_int("num_hidden_layers",1  ,5)
  neurons_per_layer = trial.suggest_int("neurons_per_layer", 8, 128, step=8)
  epochs = trial.suggest_int("epochs", 10, 50, step=10 )
  learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-1, log=True)
  dropout_rate = trial.suggest_float("dropout_rate", 0.1, 0.5, step=0.1)
  batch_size = trial.suggest_categorical("batch_size", [16, 32, 64], step=32)
  optimizer_name = trial.suggest_categorical("optimizer_name", ["Adam", "SGD", "RMSprop"])
  weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True)


  train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, pin_memory = True)
  test_loader = DataLoader(test_dataset, batch_size=128, shuffle=True, pin_memory = True)

  # model initialization
  input_dim = 784
  output_dim = 10

  model = MyNN(input_dim, output_dim, num_hidden_layers, neurons_per_layer, dropout_rate)
  model.to(device)

  #Params init
  # epochs = 100
  # learning_rate = 0.1

  # optimizer selection
    # LOSS FUNCTION
  criterion = nn.CrossEntropyLoss()
    # OPTIMIZER
  if optimizer_name == "Adam":
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay = weight_decay)
  elif optimizer_name == "SGD":
    optimizer = optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9, weight_decay = weight_decay)
  else:
    optimizer = optim.RMSprop(model.parameters(), lr=learning_rate, weight_decay = weight_decay)

  # TRAINING LOOP
  for epoch in range(epochs):
    total_epoch_loss = 0
    i = 0
    for batch_features, batch_labels in train_loader:
      ## Move the data to GPU
      ## ADDED for GPU
      batch_features = batch_features.to(device)
      batch_labels = batch_labels.to(device)

      # Forward pass
      outputs = model(batch_features)
      # print(f"for batch {i}: batch_labels = {batch_labels}")
      # print(f"for batch {i}: outputs = {outputs}")
      loss = criterion(outputs, batch_labels)

      # Backward pass and optimization
      optimizer.zero_grad()
      loss.backward()
      optimizer.step()
      total_epoch_loss += loss.item()
      i +=1
    avg_epoch_loss = total_epoch_loss/len(train_loader)
    # print(f"loss in epoch{epoch} = {avg_epoch_loss}")

  # evaluation loop
  model.eval()
  # EVALUATION CODE for TEST DATA
  total = 0
  correct = 0

  with torch.no_grad():
    for batch_features, batch_labels in test_loader:
      ## Move the data to GPU
      ## ADDED for GPU
      batch_features = batch_features.to(device)
      batch_labels = batch_labels.to(device)

      outputs = model(batch_features)
      _, predicted = torch.max(outputs.data, 1)
        # the _, is ignoring the confidence score, predicted is storing the
        # index of the highest confidence score
      total += batch_labels.size(0)
      correct += (predicted == batch_labels).sum().item()

  accuracy = 100 * correct / total
  print(f"Accuracy on the test set: {accuracy:.2f}%")

  return accuracy

In [ ]:
# !pip install optuna

In [ ]:
import optuna
study = optuna.create_study(direction='maximize')

[I 2025-07-14 13:53:25,595] A new study created in memory with name: no-name-e2384a8a-caaf-49a5-9bd6-c5b397c6d751


In [ ]:
study.optimize(objective_function, n_trials=10)

[I 2025-07-14 13:55:54,127] Trial 0 finished with value: 56.28333333333333 and parameters: {'num_hidden_layers': 5, 'neurons_per_layer': 8}. Best is trial 0 with value: 56.28333333333333.


Accuracy on the test set: 56.28%


[I 2025-07-14 13:57:36,896] Trial 1 finished with value: 87.825 and parameters: {'num_hidden_layers': 2, 'neurons_per_layer': 48}. Best is trial 1 with value: 87.825.


Accuracy on the test set: 87.83%


[I 2025-07-14 13:59:35,064] Trial 2 finished with value: 89.16666666666667 and parameters: {'num_hidden_layers': 3, 'neurons_per_layer': 128}. Best is trial 2 with value: 89.16666666666667.


...

Accuracy on the test set: 85.18%


[I 2025-07-14 14:09:18,974] Trial 8 finished with value: 88.55833333333334 and parameters: {'num_hidden_layers': 3, 'neurons_per_layer': 72}. Best is trial 2 with value: 89.16666666666667.


Accuracy on the test set: 88.56%


[I 2025-07-14 14:11:48,178] Trial 9 finished with value: 88.38333333333334 and parameters: {'num_hidden_layers': 5, 'neurons_per_layer': 96}. Best is trial 2 with value: 89.16666666666667.


Accuracy on the test set: 88.38%


In [ ]:
#To see the best parameters
study.best_params

{'num_hidden_layers': 3, 'neurons_per_layer': 128}

In [ ]:
# To see the best trial
study.best_trial

FrozenTrial(number=2, state=1, values=[89.16666666666667], datetime_start=datetime.datetime(2025, 7, 14, 13, 57, 36, 896920), datetime_complete=datetime.datetime(2025, 7, 14, 13, 59, 35, 64744), params={'num_hidden_layers': 3, 'neurons_per_layer': 128}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'num_hidden_layers': IntDistribution(high=5, log=False, low=1, step=1), 'neurons_per_layer': IntDistribution(high=128, log=False, low=8, step=8)}, trial_id=2, value=None)

In [ ]:
# To see the best value
study.best_value

89.16666666666667

In [ ]:
# Building an Objective function

# Ways to improve the acurracy :
    #GPU:
      # used larger dataset
      # created device object from cuda GPU
      # moved model to GPU
      # moved data from dataloader during training and inference to GPU
      # increased batch size during dataloader creation
      # enable the pin memory
          # i.e instead of pager to pin mem directly to pin memory
    # size of dataset,
    # optimizer
    # learning rate
    # epochs
    # weights initialization
    # regularization
    # batch normalization
    # model architecture